# Rarity-driven sparse retrieval — diagnostics

| Section | What runs | Gate |
|---|---|---|
| Step 1 | `rarity` — IDF over the pool, irregular-term list | terms are scriptural vocabulary, not tokenizer debris |
| Step 3 | `leakage` — near-duplicate audit, quarantine | flag count small enough that quarantining leaves the pool intact |
| Step 2 | `sparse_select` — greedy coverage dry run | the channel fires often enough, and is less redundant than dense |


In [1]:
# e5-large in fp32 over a 10.8k-row pool; any Colab GPU is enough.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB


In [2]:
%cd /home/prnamhr/projects/Style-Aware-MT
!pip install -r requirements.txt

# Text-only pipeline; these two carry an ABI mismatch against the pinned torch.
!pip uninstall -y torchvision torchaudio

/home/prnamhr/projects/Style-Aware-MT



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


`data/knn_index/` is git-ignored, so the pool index is rebuilt each session. The register
centroid is committed and already present.

In [3]:
!python3 manage.py build_index --config configs/base_qwen.yaml

Embedding 10860 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Batches: 100%|███████████████████████████████| 340/340 [00:13<00:00, 25.34it/s]
Wrote index to data/knn_index/ : embeddings (10860, 1024), 10860 pairs


---
## Step 1 — the rarity list

In [4]:
!python3 manage.py rarity --config configs/sparse_retrieval.yaml

Computing IDF over 10860 pool sources (zwnj=keep) ...
  22789 pool terms, 11121 at df >= 2 -> 500 frozen (requested 500)
  2.2% of the vocabulary, realized df [2, 2]
  df histogram (frozen): {'1': 0, '2': 500, '3': 0, '4-9': 0, '10-99': 0, '100+': 0}
  ZWNJ variant collisions: 25 [('آسود\u200cگی', 'آسودگی'), ('الذین\u200cهم', 'الذینهم'), ('انشآء\u200cالله', 'انشآءالله')]
Wrote results/rarity_train.json and results/rarity_train_sample50.tsv


In [5]:
import json

import pandas as pd

rarity = json.load(open('results/rarity_train.json'))
cfg = rarity['config']
print(f"{rarity['n_terms']} pool terms, {rarity['n_eligible']} at df >= {cfg['min_df']} "
      f"-> {rarity['n_frozen']} frozen (requested {cfg['freeze_n']})")
print(f"realized df {rarity['df_observed']}, {rarity['selected_frac']:.1%} of the vocabulary")
print('pool df histogram  :', rarity['df_histogram']['pool'])
print('frozen df histogram:', rarity['df_histogram']['frozen'])

sample = pd.read_csv('results/rarity_train_sample50.tsv', sep='\t')
sample['example'] = sample['example'].str.slice(0, 60)
sample

22789 pool terms, 11121 at df >= 2 -> 500 frozen (requested 500)
realized df [2, 2], 2.2% of the vocabulary
pool df histogram  : {'1': 11668, '2': 3669, '3': 1809, '4-9': 3357, '10-99': 2113, '100+': 173}
frozen df histogram: {'1': 0, '2': 500, '3': 0, '4-9': 0, '10-99': 0, '100+': 0}


,term,idf,df,example
0,آفریدگان,9.1943,2,پس گوهر پاک مردمرا از میان آفریدگان برگزید و ا...
1,أبرز,9.1943,2,وعلاوة على مواصلة حضرة ولی أمرالله شرح أهمیة ا...
2,أخیه,9.1943,2,کما وصفه حضرة الباب فی کتاب قیّوم الأسماء بأنّ...
3,أداؤها,9.1943,2,أمّا إذا حلّ وقت الصلاة وکان المسافر مستریحا و...
4,أذکر,9.1943,2,أذکر من کان أعظم منک شأناً وأکبر منک مقاماً أی...
5,أمرناکم,9.1943,2,إنا أمرناکم بکسر حدودات النفس والهوى لا ما رقم...
6,أنزلها,9.1943,2,اشتملت بعض الألواح المبارکة التی أنزلها حضرة ب...
7,اتابک,9.1943,2,میرزا تقی خان امیرنظام وزیر اعظم و اتابک معظّم...
8,اثرها,9.1943,2,فانّها حین طلوعها عن افقها تکون حرارتها و اثره...
9,اثنای,9.1943,2,و خود باب اهمّیّتی باین شور و آشوب نداده در نه...


### Normalization check — ZWNJ

In [6]:
collisions = json.load(open('results/rarity_train.json'))['zwnj_collisions']
print(f'{len(collisions)} ZWNJ variant collisions')
pd.DataFrame(collisions, columns=['split spelling', 'joined spelling']).head(25)

25 ZWNJ variant collisions


,split spelling,joined spelling
0,آسود‌گی,آسودگی
1,الذین‌هم,الذینهم
2,انشآء‌الله,انشآءالله
3,این‌قدر,اینقدر
4,بیچار‌گان,بیچارگان
5,بی‌خبر,بیخبر
6,بی‌مثال,بیمثال
7,جان‌فزا,جانفزا
8,خون‌ریزی,خونریزی
9,راست‌گو,راستگو


---
## Step 3 — leakage audit

In [7]:
!python manage.py leakage \
    --config configs/sparse_retrieval.yaml \
    --split val \
    --write-quarantine

Auditing 1323 val rows against 10860 pool rows ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 3310.87it/s]
  17/1323 eval rows flagged, 22 pool rows implicated -> results/leakage_val.json
Quarantined 22 pool rows (0.20%) -> data/splits/pool_quarantine.json


In [8]:
leak = json.load(open('results/leakage_val.json'))
print(f"val: {leak['n_eval_rows_flagged']}/{leak['n_eval_rows']} eval rows flagged, "
      f"{leak['n_pool_rows_flagged']} pool rows implicated")
print('  max-cos histogram:', leak['max_cos_histogram'])

pd.DataFrame([
    {'cos': f['cos'], 'jac_src': f['jaccard_source'], 'jac_tgt': f['jaccard_target'],
     'eval': f['eval_source'][:50], 'pool': f['pool_source'][:50]}
    for f in leak['flags'][:10]
])

val: 17/1323 eval rows flagged, 22 pool rows implicated
  max-cos histogram: {'0.00-0.50': 0, '0.50-0.70': 0, '0.70-0.80': 0, '0.80-0.85': 0, '0.85-0.90': 506, '0.90-0.95': 816, '0.95-0.97': 1, '0.97-0.99': 0, '0.99-1.01': 0}


,cos,jac_src,jac_tgt,eval,pool
0,0.9508,0.7195,0.2353,امید هست در ظلّ سدرهٴ عنایت الهی تربیت شوید و ...,امید هست در ظلّ سدرهٔ عنایت الهیّه تربیت شوید ...
1,0.9451,0.8319,0.8779,چون که هر روز را امری و هر حین را حکمی مقتضی ل...,چون که هر روز را امری و هر حین را حکمتی مقتضی ...
2,0.9394,0.7059,0.6381,تمسّکوا بحبل الأسباب متوکّلین علی الله مسبّب ا...,تمسّکوا بحبل الأسباب متوکلین على الله مسبّب ال...
3,0.9331,0.7209,0.4286,انّک انت القویّ المقتدر العزیز المتین.,انّک انت المقتدر المتعالی القویّ العزیز العظیم.
4,0.9301,0.7313,0.8491,یا حزب الله مربّی عالم عدل است چه که دارای دو ...,مربّی عالم عدلست چه که دارای دو رکن است مجازات...
5,0.9298,0.5897,0.7500,ابغض النّاس عند الله من یقعد و یطلب.,أبغض الناس عند الله من یقعد ویطلب.
6,0.9267,0.9459,0.4831,انّک انت المقتدر العزیز المهیمن القیّوم.,و انّک انت المقتدر المهیمن العزیز القیّوم.
7,0.9251,0.8611,0.3600,و لا یسأل عمّا یفعل و کلّ عن کلّ یسألون.,انّه لا یسأل عمّا یفعل و کلّ عن کلّ یسألون.
8,0.9209,0.7308,0.7674,انّ ربّک لهو العلیم الحکیم.,انّ ربّک هو العلیم الحکیم.
9,0.9205,0.9048,0.5909,و الحمد لله ربّ العالمین.,الحمد لله ربّ العالمین!


In [9]:
!python3 manage.py build_index --config configs/base_qwen.yaml \
    --index_dir data/knn_index_clean \
    --quarantine data/splits/pool_quarantine.json

Quarantine data/splits/pool_quarantine.json: dropped 22 of 10860 rows
Embedding 10838 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Batches: 100%|███████████████████████████████| 339/339 [00:13<00:00, 25.50it/s]
Wrote index to data/knn_index_clean/ : embeddings (10838, 1024), 10838 pairs


---
## Step 2 — the sparse channel

In [10]:
!python3 manage.py sparse_select --config configs/sparse_retrieval.yaml \
    --split val --index_dir data/knn_index_clean

Selecting k=8 (m=4 sparse) for 1323 val sources ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 2549.05it/s]
Dense-only baseline for the redundancy comparison ...
  routes: {'full': 0.0, 'partial': 0.0642, 'dense': 0.9358}
  rarity slots filled: mean 0.072, {'0': 1238, '1': 76, '2': 8, '3': 1, '4': 0}
  queries with >= n irregular terms: {'1': 0.0642, '2': 0.0068, '3': 0.0008, '4': 0.0, '5': 0.0, '6': 0.0}
  mean coverage (routed): 1.0
  intra-set cosine: {'sparse': 0.8995, 'dense_baseline': 0.9002}
Wrote results/sparse_selection_val.json


In [11]:
sel = json.load(open('results/sparse_selection_val.json'))
print('routes            :', sel['route_fractions'])
print('irregular terms/query — mean', sel['query_terms']['mean'],
      'deciles', sel['query_terms']['deciles'])
print('share at or above :', sel['query_terms']['share_at_or_above'])
print('coverage (routed) :', sel['coverage']['mean'])
print('intra-set cosine  :', sel['intra_set_similarity'])

routes            : {'full': 0.0, 'partial': 0.0642, 'dense': 0.9358}
irregular terms/query — mean 0.072 deciles [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3]
share at or above : {'1': 0.0642, '2': 0.0068, '3': 0.0008, '4': 0.0, '5': 0.0, '6': 0.0}
coverage (routed) : 1.0
intra-set cosine  : {'sparse': 0.8995, 'dense_baseline': 0.9002}


### Threshold sweep

`min_query_terms` is the one knob that decides whether the channel exists at all. Selection
is cheap once the index is loaded, so sweep it rather than arguing about it. Each run
writes its own report, leaving the configured run's above intact.

In [12]:
for thr in (1, 2, 3, 4):
    print(f'--- min_query_terms={thr}')
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --min_query_terms {thr} --out results/sparse_sweep_val_t{thr}.json 2>&1 | grep -E 'routes|intra-set'

--- min_query_terms=1
  routes: {'full': 0.0, 'partial': 0.0642, 'dense': 0.9358}
  intra-set cosine: {'sparse': 0.8995, 'dense_baseline': 0.9002}
--- min_query_terms=2
  routes: {'full': 0.0, 'partial': 0.0068, 'dense': 0.9932}
  intra-set cosine: {'sparse': 0.9, 'dense_baseline': 0.9002}
--- min_query_terms=3
  routes: {'full': 0.0, 'partial': 0.0008, 'dense': 0.9992}
  intra-set cosine: {'sparse': 0.9002, 'dense_baseline': 0.9002}
--- min_query_terms=4
  routes: {'full': 0.0, 'partial': 0.0, 'dense': 1.0}
  intra-set cosine: {'sparse': 0.9002, 'dense_baseline': 0.9002}


### The min_df floor

In [13]:
for min_df in (2, 10, 20):
    lst = f'results/rarity_train_mindf{min_df}.json'
    print(f'--- min_df={min_df}')
    !python3 manage.py rarity --config configs/sparse_retrieval.yaml --min_df {min_df} --out {lst} 2>&1 | grep -E 'frozen|realized'
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --rarity {lst} --out results/sparse_sweep_val_mindf{min_df}.json 2>&1 | grep -E 'routes|rarity slots'

--- min_df=2
  22789 pool terms, 11121 at df >= 2 -> 500 frozen (requested 500)
  2.2% of the vocabulary, realized df [2, 2]
  df histogram (frozen): {'1': 0, '2': 500, '3': 0, '4-9': 0, '10-99': 0, '100+': 0}
  routes: {'full': 0.0, 'partial': 0.0642, 'dense': 0.9358}
  rarity slots filled: mean 0.072, {'0': 1238, '1': 76, '2': 8, '3': 1, '4': 0}
--- min_df=10
  22789 pool terms, 2286 at df >= 10 -> 500 frozen (requested 500)
  2.2% of the vocabulary, realized df [10, 12]
  df histogram (frozen): {'1': 0, '2': 0, '3': 0, '4-9': 0, '10-99': 500, '100+': 0}
  routes: {'full': 0.0008, 'partial': 0.3023, 'dense': 0.6969}
  rarity slots filled: mean 0.386, {'0': 922, '1': 304, '2': 85, '3': 11, '4': 1}
--- min_df=20
  22789 pool terms, 1127 at df >= 20 -> 500 frozen (requested 500)
  2.2% of the vocabulary, realized df [20, 34]
  df histogram (frozen): {'1': 0, '2': 0, '3': 0, '4-9': 0, '10-99': 500, '100+': 0}
  routes: {'full': 0.0113, 'partial': 0.5019, 'dense': 0.4868}
  rarity slots f

### Worked examples

In [14]:
for ex in sel['examples']:
    print('QUERY :', ex['source'][:90])
    print('  route', ex['trace']['route'], '| terms', ex['trace']['query_terms'],
          '| coverage', ex['trace']['coverage'])
    for e in ex['exemplars']:
        print('   -', e[:90])
    print()

QUERY : جواهر الأسرار فی معارج الأسفار لمن اراد ان یتقرّب بالله المقتدر الغفّار فهنیاً للأبرار الّ
  route partial | terms ['الأسفار', 'الأنهار'] | coverage 1.0
   - و بهدایت کبری و ربوبیّت عظمی مبعوث شوند که تا قلوب مشتاقین و حقایق صافین را بالهامات غیبیّ
   - فأمطر من سحاب فیض فضلک ما تطهّر به افئدة عبادک عمّا یحجبهم عن النّظر الی وجهک و یمنعهم عن 
   - اللّهمّ انّی اسألک بالحرف الّتی اذا خرجت من فم مشیّتک ماجت البحار و هاجت الأریاح و ظهرت ال
   - من شرب من الکأس الّتی تدور بها ید رحمتک ینقطع عن دونک و ینجذب بکلمة منه عبادک الّذین رقدوا
   - لو تعرف ما نزّل من قلمی و تطّلع علی خزائن امری و لآلئ اسراری فی بحور اسمائی و اواعی کلماتی
   - طوبی لک بما کنت سائراً فی بلاد الله و کنت آیة الفرح و الاطمینان لأهل البهآء الّذین انقطعوا
   - و اسألک یا الهی باسمک الّذی به امطرت السّحاب و جرت الأنهار و اشتعلت نار الحبّ فی الأشطار ب
   - بشأن الآیة المبارکة فی الأسفار إذا نزلتم واسترحتم المقام الآمن مکان کلّ صلوة سجدة واحدة.

QUERY : و اسأل الله بأن یؤیّدنی بذلک اذ هو ارحم الرّاحمین و معطی السّائل